## 07. Modeliranje: Eksperiment C (MC + classical FE + ML)

U ovom notebook-u treniramo klasične ML modele (multi-label klasifikacija) nad Multi-Class (MC) podskupom proteina. Koristimo feature-set kombinacije definisane u notebook-u 03: AAC-CV, TF-IDF + SVD i PH.

### Uvoz biblioteka

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
import joblib

from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    f1_score, hamming_loss, accuracy_score,
    classification_report, make_scorer
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

### Učitavanje podataka

Koristimo isti MC train/test split i isti MultiLabelBinarizet koji su definisani u notebook-u 03 (nema ponovnog fitovanja).

In [2]:
DATA_DIR = "../../data/processed/localization"
FEATURES_DIR = "../../data/features/localization"
MODELS_DIR = "../../data/models/localization"
METRICS_DIR = "../../results/metrics/localization"
FIGURES_DIR = "../../results/figures/localization"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

RANDOM_STATE = 42

# Entry liste za MC train/test skup (03 notebook)
entries_mc_train = pd.read_csv(os.path.join(FEATURES_DIR, "mc_train_entries.csv"))["Entry"].values
entries_mc_test = pd.read_csv(os.path.join(FEATURES_DIR, "mc_test_entries.csv"))["Entry"].values

# Multi-hot labele (fitovane u 03 - MultiLabelBinarizer)
mlb = joblib.load(os.path.join(FEATURES_DIR, "mc_label_binarizer.pkl"))
label_columns = list(mlb.classes_)

# Labels
mc_labels_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_labels.csv"), index_col="Entry")
mc_labels_df

y_mc_train = mc_labels_df.loc[entries_mc_train, label_columns].values
y_mc_test = mc_labels_df.loc[entries_mc_test, label_columns].values

print(f"Train skup: {len(entries_mc_train)} proteina")
print(f"Test skup: {len(entries_mc_test)} proteina")
print(f"Klase: {label_columns}")
print(f"y_mc_train: {y_mc_train.shape} | y_mc_test: {y_mc_test.shape}")

Train skup: 11708 proteina
Test skup: 2858 proteina
Klase: ['Cell membrane', 'Cytoplasm', 'Mitochondrion', 'Nucleus', 'Secreted']
y_mc_train: (11708, 5) | y_mc_test: (2858, 5)


### Učitavanje raw obilježja

In [3]:
# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)
aac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")
aac_cols = list(aac_df.columns)

print(f"AAC-CV kolone: {len(aac_cols)}")
aac_df.head()

AAC-CV kolone: 20


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
Entry,,,,,,,,,,,,,,,,,,,,
A0A0C4DH62,0.058824,0.000000,0.000000,0.058824,0.058824,0.117647,0.058824,0.000000,0.000000,0.058824,0.000000,0.000000,0.000000,0.117647,0.000000,0.117647,0.117647,0.117647,0.058824,0.058824
A0A0A0MT87,0.125000,0.000000,0.000000,0.000000,0.062500,0.125000,0.000000,0.062500,0.062500,0.125000,0.000000,0.062500,0.000000,0.062500,0.062500,0.062500,0.062500,0.062500,0.000000,0.062500
P01880,0.074419,0.018605,0.034884,0.062791,0.027907,0.046512,0.020930,0.023256,0.048837,0.097674,0.011628,0.023256,0.095349,0.053488,0.048837,0.100000,0.097674,0.065116,0.025581,0.023256
P01876,0.062814,0.035176,0.032663,0.052764,0.035176,0.060302,0.017588,0.010050,0.030151,0.115578,0.005025,0.022613,0.115578,0.042714,0.035176,0.103015,0.118090,0.065327,0.020101,0.020101
P01877,0.066496,0.038363,0.035806,0.056266,0.033248,0.061381,0.020460,0.010230,0.030691,0.112532,0.007673,0.025575,0.104859,0.046036,0.038363,0.092072,0.109974,0.069054,0.020460,0.020460


In [4]:
# Sirove sekvence (fitovanje TF-IDF+SVD dešava se unutar Pipeline-a, po CV foldu)
mc_train_seq_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_train_sequences_raw.csv"), index_col="Entry")
mc_test_seq_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_test_sequences_raw.csv"), index_col="Entry")
seq_col = "Sequence"

print(f"Sirove sekvence — train: {mc_train_seq_df.shape[0]} | test: {mc_test_seq_df.shape[0]}")
mc_train_seq_df.head()

Sirove sekvence — train: 11708 | test: 2858


,Sequence
Entry,
A0A0C4DH62,AEYFQHWGQGTLVTVSS
A0A0A0MT87,AKNIQYFGAGTRLSVL
P01880,APTKAPDVFPIISGCRHPKDNSPVVLACLITGYHPTSVTVTWYMGT...
P01876,ASPTSPKVFPLSLCSTQPDGNVVIACLVQGFFPQEPLSVTWSESGQ...
P01859,ASTKGPSVFPLAPCSRSTSESTAALGCLVKDYFPEPVTVSWNSGAL...


In [5]:
# Sirove (neskalirane) physchem osobine (StandardScaler fituje se unutar Pipeline-a, po CV foldu)
mc_train_ph_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_train_physchem_raw.csv"), index_col="Entry")
mc_test_ph_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_test_physchem_raw.csv"), index_col="Entry")
ph_cols = list(mc_train_ph_df.columns)

print(f"PH kolone: {ph_cols}")
mc_train_ph_df.head()

PH kolone: ['MW', 'pI', 'GRAVY', 'Aromaticity', 'Instability']


,MW,pI,GRAVY,Aromaticity,Instability
Entry,,,,,
A0A0C4DH62,1910.0464,5.241828,-0.170588,0.176471,2.958824
A0A0A0MT87,1737.9955,9.994937,0.231250,0.125000,21.512500
P01880,47499.2162,7.251087,-0.430465,0.076744,64.091628
P01876,42848.0149,5.426726,-0.210302,0.075377,58.911332
P01859,43805.2643,6.111124,-0.270886,0.093671,45.856481


### Sastavljanje DataFrame-a 

Spajamo AAC-CV, raw sekvence i neskalirane PH vrijednosti u jedan DataFrame po Entry-ju, poravnat sa redoslijedom labela (y_sc_train/y_sc_test).

In [6]:
X_train_raw = aac_df.join(mc_train_seq_df, how="inner").join(mc_train_ph_df, how="inner") 
X_test_raw = aac_df.join(mc_test_seq_df, how="inner").join(mc_test_ph_df, how="inner") 

# Poravnanje redoslijeda redova sa redoslijedom labela
X_train_raw = X_train_raw.loc[entries_mc_train]
X_test_raw = X_test_raw.loc[entries_mc_test]

assert list(X_train_raw.index) == list(entries_mc_train)
assert list(X_test_raw.index) == list(entries_mc_test)

print(f"X_train_raw: {X_train_raw.shape} | X_test_raw: {X_test_raw.shape}")
X_train_raw.head()

X_train_raw: (11708, 26) | X_test_raw: (2858, 26)


,A,C,D,E,F,G,H,I,K,L,...,T,V,W,Y,Sequence,MW,pI,GRAVY,Aromaticity,Instability
Entry,,,,,,,,,,,,,,,,,,,,,
A0A0C4DH62,0.058824,0.000000,0.000000,0.058824,0.058824,0.117647,0.058824,0.000000,0.000000,0.058824,...,0.117647,0.117647,0.058824,0.058824,AEYFQHWGQGTLVTVSS,1910.0464,5.241828,-0.170588,0.176471,2.958824
A0A0A0MT87,0.125000,0.000000,0.000000,0.000000,0.062500,0.125000,0.000000,0.062500,0.062500,0.125000,...,0.062500,0.062500,0.000000,0.062500,AKNIQYFGAGTRLSVL,1737.9955,9.994937,0.231250,0.125000,21.512500
P01880,0.074419,0.018605,0.034884,0.062791,0.027907,0.046512,0.020930,0.023256,0.048837,0.097674,...,0.097674,0.065116,0.025581,0.023256,APTKAPDVFPIISGCRHPKDNSPVVLACLITGYHPTSVTVTWYMGT...,47499.2162,7.251087,-0.430465,0.076744,64.091628
P01876,0.062814,0.035176,0.032663,0.052764,0.035176,0.060302,0.017588,0.010050,0.030151,0.115578,...,0.118090,0.065327,0.020101,0.020101,ASPTSPKVFPLSLCSTQPDGNVVIACLVQGFFPQEPLSVTWSESGQ...,42848.0149,5.426726,-0.210302,0.075377,58.911332
P01859,0.037975,0.032911,0.040506,0.060759,0.045570,0.045570,0.020253,0.027848,0.065823,0.073418,...,0.086076,0.106329,0.017722,0.030380,ASTKGPSVFPLAPCSRSTSESTAALGCLVKDYFPEPVTVSWNSGAL...,43805.2643,6.111124,-0.270886,0.093671,45.856481


### Definisanje feature-ser kombinacija (kroz ColumnTransfer)

In [7]:
# Funkcija koja bira odgovarajuci transformator i vraca Pipeline
# (svaka feature-set/model kombinacija ima sopstveni transformator)
def make_tfidf_svd_step():
    return Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char", ngram_range=(2, 3), max_features=3000, lowercase=False)),
        ("svd", TruncatedSVD(n_components=150, random_state=RANDOM_STATE)),
    ])

# Funkcija koja vraca ColumnTransfer za odgovarajucu feature-set kombinaciju
def make_preprocessor(feature_set_name):
    if feature_set_name == "AAC-CV":
        transformers = [("aac", "passthrough", aac_cols)]
    elif feature_set_name == "TFIDF-SVD":
        transformers = [("tfidf-svd", make_tfidf_svd_step(), seq_col)]
    elif feature_set_name == "PH":
        transformers = [("ph_scaled", StandardScaler(), ph_cols)]
    elif feature_set_name == "AAC-CV+TFIDF-SVD":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
        ]
    elif feature_set_name == "AAC-CV+PH":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    elif feature_set_name == "TFIDF-SVD+PH":
        transformers = [
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    elif feature_set_name == "AAC-CV+TFIDF-SVD+PH":
        transformers = [
            ("aac", "passthrough", aac_cols),
            ("tfidf-svd", make_tfidf_svd_step(), seq_col),
            ("ph_scaled", StandardScaler(), ph_cols),
        ]
    else:
        raise ValueError(f"Nepoznat feature set: {feature_set_name}")

    return ColumnTransformer(transformers=transformers, remainder="drop")

FEATURE_SET_NAMES = [
    "AAC-CV",
    "TFIDF-SVD",
    "PH",
    "AAC-CV+TFIDF-SVD",
    "AAC-CV+PH",
    "TFIDF-SVD+PH",
    "AAC-CV+TFIDF-SVD+PH",
]


### AAC-CV

In [8]:
"""# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)
# Selektujemo redove koji pripadaju MC train/test skupu
aac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")

mc_train_aac = aac_df.loc[entries_mc_train].values
mc_test_aac = aac_df.loc[entries_mc_test].values

print(f"AAC-CV train: {mc_train_aac.shape} | test: {mc_test_aac.shape}")"""

'# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)\n# Selektujemo redove koji pripadaju MC train/test skupu\naac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")\n\nmc_train_aac = aac_df.loc[entries_mc_train].values\nmc_test_aac = aac_df.loc[entries_mc_test].values\n\nprint(f"AAC-CV train: {mc_train_aac.shape} | test: {mc_test_aac.shape}")'

### TF-IDF + SVD

In [9]:
"""# Fitovano na MC train skupu u 03
mc_train_tfidf = np.load(os.path.join(FEATURES_DIR, "mc_train_tfidf_svd.npy")) 
mc_test_tfidf = np.load(os.path.join(FEATURES_DIR, "mc_test_tfidf_svd.npy")) 

print(f"TF-IDF-SVD train: {mc_train_tfidf.shape} | test: {mc_test_tfidf.shape}")"""

'# Fitovano na MC train skupu u 03\nmc_train_tfidf = np.load(os.path.join(FEATURES_DIR, "mc_train_tfidf_svd.npy")) \nmc_test_tfidf = np.load(os.path.join(FEATURES_DIR, "mc_test_tfidf_svd.npy")) \n\nprint(f"TF-IDF-SVD train: {mc_train_tfidf.shape} | test: {mc_test_tfidf.shape}")'

### Fizičko-hemijske osobine

In [10]:
"""mc_train_ph = np.load(os.path.join(FEATURES_DIR, "mc_train_physchem_scaled.npy"))
mc_test_ph = np.load(os.path.join(FEATURES_DIR, "mc_test_physchem_scaled.npy"))

print(f"PH train: {mc_train_ph.shape} | test: {mc_test_ph.shape}")"""

'mc_train_ph = np.load(os.path.join(FEATURES_DIR, "mc_train_physchem_scaled.npy"))\nmc_test_ph = np.load(os.path.join(FEATURES_DIR, "mc_test_physchem_scaled.npy"))\n\nprint(f"PH train: {mc_train_ph.shape} | test: {mc_test_ph.shape}")'

### Kombinovanje u feature setove

In [11]:
"""feature_sets_train = {
    "AAC-CV": mc_train_aac,
    "TFIDF-SVD": mc_train_tfidf,
    "PH": mc_train_ph,
    "AAC-CV+TFIDF-SVD": np.hstack([mc_train_aac, mc_train_tfidf]),
    "AAC-CV+PH": np.hstack([mc_train_aac, mc_train_ph]),
    "TFIDF-SVD+PH": np.hstack([mc_train_tfidf, mc_train_ph]),
    "AAC-CV+TFIDF-SVD+PH": np.hstack([mc_train_aac, mc_train_tfidf, mc_train_ph]),
}

feature_sets_test = {
    "AAC-CV": mc_test_aac,
    "TFIDF-SVD": mc_test_tfidf,
    "PH": mc_test_ph,
    "AAC-CV+TFIDF-SVD": np.hstack([mc_test_aac, mc_test_tfidf]),
    "AAC-CV+PH": np.hstack([mc_test_aac, mc_test_ph]),
    "TFIDF-SVD+PH": np.hstack([mc_test_tfidf, mc_test_ph]),
    "AAC-CV+TFIDF-SVD+PH": np.hstack([mc_test_aac, mc_test_tfidf, mc_test_ph]),
}

for name, matrix in feature_sets_train.items():
    print(f"{name:<20} train: {matrix.shape} test: {feature_sets_test[name].shape}")"""

'feature_sets_train = {\n    "AAC-CV": mc_train_aac,\n    "TFIDF-SVD": mc_train_tfidf,\n    "PH": mc_train_ph,\n    "AAC-CV+TFIDF-SVD": np.hstack([mc_train_aac, mc_train_tfidf]),\n    "AAC-CV+PH": np.hstack([mc_train_aac, mc_train_ph]),\n    "TFIDF-SVD+PH": np.hstack([mc_train_tfidf, mc_train_ph]),\n    "AAC-CV+TFIDF-SVD+PH": np.hstack([mc_train_aac, mc_train_tfidf, mc_train_ph]),\n}\n\nfeature_sets_test = {\n    "AAC-CV": mc_test_aac,\n    "TFIDF-SVD": mc_test_tfidf,\n    "PH": mc_test_ph,\n    "AAC-CV+TFIDF-SVD": np.hstack([mc_test_aac, mc_test_tfidf]),\n    "AAC-CV+PH": np.hstack([mc_test_aac, mc_test_ph]),\n    "TFIDF-SVD+PH": np.hstack([mc_test_tfidf, mc_test_ph]),\n    "AAC-CV+TFIDF-SVD+PH": np.hstack([mc_test_aac, mc_test_tfidf, mc_test_ph]),\n}\n\nfor name, matrix in feature_sets_train.items():\n    print(f"{name:<20} train: {matrix.shape} test: {feature_sets_test[name].shape}")'

### Definisanje modela i mreže hiperparametara

In [19]:
param_grids = {
    "LogisticRegression": {
        "model": OneVsRestClassifier(
            LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
        ),
        "params": {
            "estimator__C": [0.01, 0.1, 1, 10, 100],
        },
    },
    "RandomForest": {
        "model": OneVsRestClassifier(
            RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE)
        ),
        "params": {
            "estimator__n_estimators": [100, 200],
            "estimator__max_depth": [10, 25],
            "estimator__min_samples_leaf": [2, 5],
        },
    },
    "SVM": {
        "model": OneVsRestClassifier(
            SVC(class_weight="balanced", random_state=RANDOM_STATE)
        ),
        "params": {
            "estimator__C": [0.1, 1, 10],
            "estimator__kernel": ["rbf", "linear"],
        },
    },
}

f1_micro_scorer = make_scorer(f1_score, average="micro", zero_division=0)
cv = MultilabelStratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

### Glavna petlja treniranja

In [20]:
from joblib import Memory

cache_dir = "../cache/pipeline_cache"
memory = Memory(location=cache_dir, verbose=0)

all_results = []

for feature_name in FEATURE_SET_NAMES:
    preprocessor = make_preprocessor(feature_name)

    for model_name, config in param_grids.items():
        print(f"Treniranje: {model_name:<20} | Feature set: {feature_name}")

        full_pipeline = Pipeline([
            ("features", preprocessor),
            ("model", config["model"]),
        ], memory=memory)

        pipeline_param_grid = {
            f"model__{param_name}": values 
            for param_name, values in config["params"].items()
        }

        grid = GridSearchCV(
            estimator=full_pipeline,
            param_grid=pipeline_param_grid,
            cv=cv,
            scoring=f1_micro_scorer,
            n_jobs=-1,
        )

        grid.fit(X_train_raw, y_mc_train)

        best_pipeline = grid.best_estimator_

        # Predikcije na train skupu (isti fit-ovan model, bez dodatnog treniranja)
        y_train_pred = best_pipeline.predict(X_train_raw)
        train_f1_micro = f1_score(y_mc_train, y_train_pred, average="micro", zero_division=0)
        train_f1_macro = f1_score(y_mc_train, y_train_pred, average="macro", zero_division=0)
        train_acc = accuracy_score(y_mc_train, y_train_pred) # subser accuracy
                
        # Predikcije na test skupu
        y_test_pred = best_pipeline.predict(X_test_raw)
        test_f1_micro = f1_score(y_mc_test, y_test_pred, average="micro", zero_division=0)
        test_f1_macro = f1_score(y_mc_test, y_test_pred, average="macro", zero_division=0)
        test_f1_samples = f1_score(y_mc_test, y_test_pred, average="samples", zero_division=0)
        test_acc = accuracy_score(y_mc_test, y_test_pred)  # subset accuracy (exact match)
        test_hamming = hamming_loss(y_mc_test, y_test_pred)
        
        best_params = {
            k.replace("model__", ""): v for k, v in grid.best_params_.items()
        }

        all_results.append({
            "Feature set": feature_name,
            "Model": model_name,
            "Best params": grid.best_params_,

            "CV F1 micro": grid.best_score_,

            "Train F1 micro": train_f1_micro,
            "Train F1 macro": train_f1_macro,
            "Train Subset Accuracy": train_acc,

            "Test F1 micro": test_f1_micro,
            "Test F1 macro": test_f1_macro,
            "Test F1 samples": test_f1_samples,
            "Test Subset Accuracy": test_acc,
            "Test Hamming Loss": test_hamming,
            
            "Overfit Gap (F1 micro)": train_f1_micro - test_f1_micro,
        })

        model_filename = f"mc_{model_name}_{feature_name}.pkl"
        joblib.dump(best_pipeline, os.path.join(MODELS_DIR, model_filename))

    memory.clear()

print("\nTreniranje završeno.")

Treniranje: LogisticRegression   | Feature set: AAC-CV
Treniranje: RandomForest         | Feature set: AAC-CV
Treniranje: SVM                  | Feature set: AAC-CV


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: TFIDF-SVD
Treniranje: RandomForest         | Feature set: TFIDF-SVD
Treniranje: SVM                  | Feature set: TFIDF-SVD


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: PH


...

Treniranje: LogisticRegression   | Feature set: TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: TFIDF-SVD+PH
Treniranje: SVM                  | Feature set: TFIDF-SVD+PH


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache


Treniranje: LogisticRegression   | Feature set: AAC-CV+TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: AAC-CV+TFIDF-SVD+PH
Treniranje: SVM                  | Feature set: AAC-CV+TFIDF-SVD+PH


[Memory(location=../cache/pipeline_cache\joblib)]: Flushing completely the cache



Treniranje završeno.


In [ ]:
"""all_results = []

for feature_name, X_train in feature_sets_train.items():
    X_test = feature_sets_test[feature_name]

    for model_name, config in param_grids.items():
        print(f"Treniranje: {model_name:<20} | Feature set: {feature_name}")

        grid = GridSearchCV(
            estimator=config["model"],
            param_grid=config["params"],
            cv=cv,
            scoring=f1_micro_scorer,
            n_jobs=-1,
            refit=True,
        )
        grid.fit(X_train, y_mc_train)

        best_model = grid.best_estimator_

        # Predikcije na train skupu (isti fit-ovan model, bez dodatnog treniranja)
        y_train_pred = best_model.predict(X_train)
        train_f1_micro = f1_score(y_mc_train, y_train_pred, average="micro", zero_division=0)
        train_f1_macro = f1_score(y_mc_train, y_train_pred, average="macro", zero_division=0)
        train_acc = accuracy_score(y_mc_train, y_train_pred)  # subset accuracy (exact match)

        # Predikcije na test skupu
        y_test_pred = best_model.predict(X_test)
        test_f1_micro = f1_score(y_mc_test, y_test_pred, average="micro", zero_division=0)
        test_f1_macro = f1_score(y_mc_test, y_test_pred, average="macro", zero_division=0)
        test_f1_samples = f1_score(y_mc_test, y_test_pred, average="samples", zero_division=0)
        test_acc = accuracy_score(y_mc_test, y_test_pred)  # subset accuracy (exact match)
        test_hamming = hamming_loss(y_mc_test, y_test_pred)

        all_results.append({
            "Feature set": feature_name,
            "Model": model_name,
            "Best params": grid.best_params_,

            "CV F1 micro": grid.best_score_,

            "Train F1 micro": train_f1_micro,
            "Train F1 macro": train_f1_macro,
            "Train Subset Accuracy": train_acc,

            "Test F1 micro": test_f1_micro,
            "Test F1 macro": test_f1_macro,
            "Test F1 samples": test_f1_samples,
            "Test Subset Accuracy": test_acc,
            "Test Hamming Loss": test_hamming,
            
            "Overfit Gap (F1 micro)": train_f1_micro - test_f1_micro,
        })

        model_filename = f"mc_{model_name}_{feature_name}.pkl"
        joblib.dump(best_model, os.path.join(MODELS_DIR, model_filename))

print("\nTreniranje završeno.")"""

Treniranje: LogisticRegression   | Feature set: AAC-CV


Treniranje: RandomForest         | Feature set: AAC-CV
Treniranje: SVM                  | Feature set: AAC-CV
Treniranje: LogisticRegression   | Feature set: TFIDF-SVD
Treniranje: RandomForest         | Feature set: TFIDF-SVD
Treniranje: SVM                  | Feature set: TFIDF-SVD
Treniranje: LogisticRegression   | Feature set: PH
Treniranje: RandomForest         | Feature set: PH
Treniranje: SVM                  | Feature set: PH
Treniranje: LogisticRegression   | Feature set: AAC-CV+TFIDF-SVD
Treniranje: RandomForest         | Feature set: AAC-CV+TFIDF-SVD
Treniranje: SVM                  | Feature set: AAC-CV+TFIDF-SVD
Treniranje: LogisticRegression   | Feature set: AAC-CV+PH
Treniranje: RandomForest         | Feature set: AAC-CV+PH
Treniranje: SVM                  | Feature set: AAC-CV+PH
Treniranje: LogisticRegression   | Feature set: TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: TFIDF-SVD+PH
Treniranje: SVM                  | Feature set: TFIDF-SVD+PH
Treniranje:

c:\Users\Korisnik\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



Treniranje završeno.


### Pregled i čuvanje rezultata

In [21]:
results_df = pd.DataFrame(all_results)

# Sortiranje po Test F1 micro
results_df = results_df.sort_values(by="CV F1 micro", ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(METRICS_DIR, "mc_classical_modeling_results.csv"), index=False)

display_cols = [
    "Model", "Feature set",
    "Train F1 micro", "Train Subset Accuracy",
    "CV F1 micro",
    "Test F1 micro", "Test F1 macro", "Test F1 samples",
    "Test Subset Accuracy", "Test Hamming Loss",
    "Overfit Gap (F1 micro)",
]

results_df[display_cols].style.format({
    "Train F1 micro": "{:.3f}",
    "Train Subset Accuracy": "{:.3f}",
    "CV F1 micro": "{:.3f}",
    "Test F1 micro": "{:.3f}",
    "Test F1 macro": "{:.3f}",
    "Test F1 samples": "{:.3f}",
    "Test Subset Accuracy": "{:.3f}",
    "Test Hamming Loss": "{:.3f}",
    "Overfit Gap (F1 micro)": "{:.3f}",
}).background_gradient(subset=["Overfit Gap (F1 micro)"], cmap="Reds")

,Model,Feature set,Train F1 micro,Train Subset Accuracy,CV F1 micro,Test F1 micro,Test F1 macro,Test F1 samples,Test Subset Accuracy,Test Hamming Loss,Overfit Gap (F1 micro)
0,SVM,AAC-CV+TFIDF-SVD,0.955,0.888,0.687,0.691,0.656,0.682,0.444,0.166,0.264
1,SVM,TFIDF-SVD,0.966,0.916,0.683,0.687,0.652,0.676,0.443,0.166,0.279
2,SVM,AAC-CV+TFIDF-SVD+PH,0.783,0.530,0.668,0.674,0.635,0.684,0.380,0.193,0.109
3,SVM,TFIDF-SVD+PH,0.783,0.530,0.667,0.674,0.634,0.683,0.378,0.193,0.109
4,RandomForest,AAC-CV+TFIDF-SVD+PH,0.858,0.672,0.657,0.659,0.615,0.642,0.391,0.183,0.199
5,RandomForest,TFIDF-SVD+PH,0.820,0.599,0.654,0.659,0.614,0.647,0.386,0.187,0.161
6,RandomForest,AAC-CV+TFIDF-SVD,0.840,0.635,0.652,0.655,0.614,0.635,0.377,0.187,0.185
7,RandomForest,TFIDF-SVD,0.814,0.589,0.651,0.651,0.604,0.637,0.378,0.192,0.163
8,RandomForest,AAC-CV+PH,0.807,0.575,0.650,0.651,0.612,0.644,0.375,0.195,0.156
9,SVM,AAC-CV,0.725,0.436,0.640,0.647,0.604,0.659,0.318,0.222,0.078
